In [0]:
from pyspark.sql import functions as F
CATALOG = "airbnb_obs"

In [0]:
# DIM HOST
silver_h = spark.table(f"{CATALOG}.silver.hosts")

dim_hosts = (silver_h.select(
              "host_id",
              "host_name",
              "is_superhost",
              "response_rate",
              "host_since",
    ).dropDuplicates(["host_id"]) # guarantee one row per host

)

(dim_hosts.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.dim_hosts"))

print(f"dim_hosts: {dim_hosts.count()}")
print(f"dim_hosts: {dim_hosts.show()}")
  

In [0]:
# DIM LISTINGS

silver_l = spark.table(f"{CATALOG}.silver.listings")

dim_listings = (silver_l.select(
              "listing_id",
              "host_id",
              "property_type",
              "city",
              "country",
              "accommodates",
              "bedrooms",
              "bathrooms", 
              "price_per_night",
    ).dropDuplicates(["listing_id"]) # guarantee one row per listing

)

(dim_listings.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.dim_listings"))

print(f"dim_listings: {dim_listings.count()}")
print(f"dim_listings: {dim_listings.show()}")
  

In [0]:
#FACT BOOKINGS

silver_b = spark.table(f"{CATALOG}.silver.bookings")

fact_bookings = (silver_b.select(
              "booking_id",
              "listing_id",
              "booking_date",
              "booking_status",
              "nights_booked",
              "booking_amount",
              "cleaning_fee",
              "service_fee",
    ).dropDuplicates(["booking_id"]) # guarantee one row per booking

)

(fact_bookings.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.fact_bookings"))

print(f"fact_bookings: {fact_bookings.count()}")
print(f"fact_bookings: {fact_bookings.show()}")
  

In [0]:
# REFERENTIAL INTEGRITY: fact ---> dim_listings

fact = spark.table(f"{CATALOG}.gold.fact_bookings")
dim = spark.table(f"{CATALOG}.gold.dim_listings")

#bookings whose listing_id has no match in dim_listings
orphans = (fact.join(dim, on="listing_id", how="left_anti"))

orphan_count = orphans.count()
print(f"orphaned bookings (listing_id not in dim_listings):{orphan_count}")

orphans.select("booking_id", "listing_id").show(12, truncate=False)

In [0]:
# INJECT AN ORPHAN (Gold integrity demo) 
fact = spark.table(f"{CATALOG}.gold.fact_bookings")

# one fake booking pointing at a listing_id that does NOT exist
bad_row = fact.limit(1).withColumn("booking_id", F.lit("ORPHAN_TEST")) \
                       .withColumn("listing_id", F.lit(999999))

fact_broken = fact.unionByName(bad_row)

(fact_broken.write.mode("overwrite").option("overwriteSchema","true")
            .saveAsTable(f"{CATALOG}.gold.fact_bookings_broken"))

print("injected 1 orphan into fact_bookings_broken")

In [0]:
fact = spark.table(f"{CATALOG}.gold.fact_bookings_broken")
dim  = spark.table(f"{CATALOG}.gold.dim_listings")

orphans = fact.join(dim, on="listing_id", how="left_anti")
print(f"orphaned bookings: {orphans.count()}")
orphans.select("booking_id","listing_id").show(10, truncate=False)

In [0]:
# ---------- LOG GOLD INTEGRITY VERDICT TO dq_results Fact table----------
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

fact = spark.table(f"{CATALOG}.gold.fact_bookings")   # the REAL table
dim  = spark.table(f"{CATALOG}.gold.dim_listings")

orphan_count = fact.join(dim, on="listing_id", how="left_anti").count()
status = "PASS" if orphan_count == 0 else "FAIL"
total_count = fact.count()

run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")  # real, sortable

schema = StructType([
    StructField("run_id",         StringType(),    True),
    StructField("check_ts",       TimestampType(), True),
    StructField("layer",          StringType(),    True),
    StructField("table_name",     StringType(),    True),
    StructField("column_name",    StringType(),    True),
    StructField("rule_name",      StringType(),    True),
    StructField("rule_type",      StringType(),    True),
    StructField("failed_records", LongType(),      True),
    StructField("total_records",  LongType(),      True),
    StructField("status",         StringType(),    True),
    StructField("details",        StringType(),    True),
])

verdict = spark.createDataFrame(
    [(
        run_id,
        datetime.now(timezone.utc),  # check_ts
        "gold",  # layer
        "fact_bookings",  # table_name
        None,  # column_name
        "referential_integrity_listing_id",  # rule_name
        "referential_integrity",  # rule_type
        orphan_count,  # failed_records
        total_count,  # total_records
        status,  # status
        None,  # details
    )],
    schema
)

(verdict.write.mode("append").saveAsTable(f"{CATALOG}.monitoring.dq_results"))

print(f"logged Gold verdict: {status}, {orphan_count} orphans")

In [0]:
# ---------- LOG GOLD INTEGRITY VERDICT TO dq_results dim tables----------

from pyspark.sql import functions as F
from datetime import datetime

run_id = datetime.utcnow().strftime("%Y%m%d%H%M%S")  # real, sortable

def check_dim_unique(table_name, key_col): 
    df = spark.table(f"{CATALOG}.gold.{table_name}")
    dupes = (df.groupBy(key_col).count().filter(F.col("count") > 1).count())
    total = df.count()
    status = "PASS" if dupes == 0  else "FAIL"

    verdict = spark.createDataFrame(
        [(run_id, datetime.utcnow(), "gold", table_name, key_col,"dimension_key_unique", "uniqueness", int(dupes), int(total), status, None)],
        schema=spark.table(f"{CATALOG}.monitoring.dq_results").schema)
    
    verdict.write.mode("append").saveAsTable(f"{CATALOG}.monitoring.dq_results")
    print(f"{key_col} -> {status}, {dupes} duplicate keys")

check_dim_unique("dim_listings", "listing_id")
check_dim_unique ("dim_hosts","host_id")

In [0]:
%sql --- CHECK CURRENT DATA QUALITY ACCROSS ALL 3 LAYERS
SELECT layer, table_name, rule_name, status
FROM airbnb_obs.monitoring.v_current_dq_state
ORDER BY layer, table_name;

In [0]:
%sql --CHECK CURRENT DATA QUALITY 
SELECT * FROM airbnb_obs.monitoring.v_current_dq_state WHERE table_name = 'fact_bookings';

In [0]:
%sql
SELECT layer, table_name, rule_name, status
FROM airbnb_obs.monitoring.v_current_dq_state
ORDER BY layer, table_name;